<div align="center" style="padding:40px 24px;background:linear-gradient(135deg,#0e1a2e 0%,#13223b 100%);border-radius:14px;border-bottom:5px solid #ffd43b;">
  <p style="margin:0;color:#6aaef0;font-family:monospace;font-weight:700;font-size:14px;letter-spacing:.16em;text-transform:uppercase;">The AI Learning Hub · Lesson 03 · Applied AI: unstructured data</p>
  <h1 style="margin:14px 0 6px;color:#f1f5fb;font-size:42px;">🎚️ Challenge 1 · Flip It!</h1>
  <p style="margin:0;color:#b4c2d8;font-size:18px;">Change as few words as possible to make a sentiment AI change its mind.</p>
</div>

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/the-ai-learning-hub/03-applied-ai-projects/blob/main/01_text_mining/03_challenge_flip_it/Challenge_1_Flip_It.ipynb) [![Slides](https://img.shields.io/badge/slides-Lesson_03-ffd43b)](https://the-ai-learning-hub.github.io/03-applied-ai-projects/#challenge-1)

In notebook 02 you trained a sentiment model on 39,000 tweets. Today you **break** it, one word at a time.
You get a sentence the model is sure about (*"I love this sunny day"* → 85 % positive). Edit it. Every keystroke is scored **live**: each word lights up green or red by how hard it pushes the model, a big gauge slides between NEGATIVE and POSITIVE, and when the verdict flips… 🎉. The fewer words you change, the shinier the medal.

## 🎯 Objectives
By the end of this challenge you will be able to:
- explain how a **linear** text model decides: a starting push (the bias) plus one push per word;
- find the **blind spots** of a bag of words: negation, slang, word order, unknown words;
- explain why a model trained on 2009 tweets has strange ideas about words like *worry*, *exam* or *sick*.

**⏱️ Duration:** ≈ 15 min · **📋 Prerequisites:** notebook [01 · SMS spam](../01_mini_project_sms_spam/01_Text_Mining_Basics_SMS_Spam.ipynb) (TF-IDF, Logistic Regression) · **▶️ How to use:** `Run All` once (≈ 10 s), scroll to **🎚️ Play!** and type. You play alone: your score is yours.

## 🎮 How to play

> **Flip the verdict with as few word swaps as possible.** Each word you **add**, **remove** or **replace** = **1 swap** (capital letters do not count; `!` and `,` are words too).
> The model is **TF-IDF + Logistic Regression**, trained in this notebook in about a second.
>
> | Level | Start sentence | Flip it to | Twist | 🥇 | 🥈 |
> |---|---|---|---|---|---|
> | 1 · 🌞 Warm-up | *I love this sunny day* | NEGATIVE | none | 1 swap | ≤ 2 |
> | 2 · 🚑 Rescue mission | *this is the worst headache ever* | POSITIVE | the obvious word is not the one | 1 | ≤ 3 |
> | 3 · 🎭 Liar mode | *this new phone is awesome* | NEGATIVE | a **human** must still read it as happy (honour box + no obvious sad words) | 1 | ≤ 3 |
> | 4 · ➕ Add-only | *I lost my keys and missed the bus* | POSITIVE | you may only **add** words | 3 | ≤ 5 |
> | 5 · 💀 Boss | *happy birthday! thank you so much, …* (99.98 %) | NEGATIVE | the model has never been so sure | 3 | ≤ 5 |
>
> Any flip that follows the rules is at least 🥉. **Stars:** 🥇 = 3 ⭐ · 🥈 = 2 ⭐ · 🥉 = 1 ⭐ per level, your **best** try counts.
>
> **Medals:** 🥉 Bronze ≥ 5 ⭐ · 🥈 Silver ≥ 9 ⭐ · 🥇 Gold ≥ 13 ⭐ (out of 15). Every gold is possible: we checked.

## 📑 Contents
1. [🤖 Meet the model](#sec-1)
2. [🎚️ Play!](#sec-2)
3. [⌨️ No widgets? Play with one line of code](#sec-3)
4. [🏆 Your scorecard](#sec-4)
5. [🔍 Why did it flip?](#sec-5)
6. [📌 Summary](#sec-6)
7. [📚 Learn more](#sec-7)

<a id="sec-0"></a>
## ⚙️ Setup
No big download: the tweets (a few MB) come from the lesson's data release the first time, and the model trains in about a second on a laptop CPU.

In [1]:
# ⚙️ Setup: works locally (VS Code / Jupyter) and in Google Colab
import os
import subprocess
import sys
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content/03-applied-ai-projects")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/the-ai-learning-hub/03-applied-ai-projects.git", str(ROOT)], check=True)
    os.chdir(ROOT)
else:  # locally: walk up until we find the folder that has src/ and data/
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir() and (p / "data").is_dir())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

print("Project root:", ROOT.name, "(Colab)" if IN_COLAB else "(local)")

Project root: 03-applied-ai-projects (local)


In [2]:
import time

import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

from src import challenges as ch
from src.paths import data_dir
from src.text import clean_tweet, tokenize

print("Ready · CPU is enough for this challenge")

Ready · CPU is enough for this challenge


<a id="sec-1"></a>
# 1. 🤖 Meet the model

The same recipe as notebook 02, kept as simple as possible: clean the tweets, turn each one into **TF-IDF** numbers (one per word), fit a **logistic regression**. Only single words (no word pairs), so each word has exactly **one weight** and we can show you its push.

In [3]:
t0 = time.perf_counter()
tweets = pd.read_csv(data_dir("tweets") / "tweets" / "tweets_train.csv")
tweets["clean"] = tweets["text"].map(clean_tweet)
tweets = tweets[tweets["clean"].str.strip() != ""].drop_duplicates("clean")    # same cleaning as notebook 02

vectorizer = TfidfVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False, min_df=2, sublinear_tf=True)
model = LogisticRegression(C=2, class_weight="balanced", max_iter=2000)
model.fit(vectorizer.fit_transform(tweets["clean"]), tweets["class"])

test = pd.read_csv(data_dir("tweets") / "tweets" / "tweets_test.csv")
pred = model.predict(vectorizer.transform(test["text"].map(clean_tweet)))
print(f"Trained on {len(tweets):,} tweets in {time.perf_counter() - t0:.1f} s · "
      f"{len(vectorizer.vocabulary_):,} words · macro-F1 on the test tweets: {f1_score(test['class'], pred, average='macro'):.2f}")

Trained on 39,152 tweets in 1.3 s · 13,975 words · macro-F1 on the test tweets: 0.70


A decent model (notebook 02 got about the same). Now the game: `FlipGame` lives in [`src/challenges.py`](../../src/challenges.py) and only needs the vectorizer and the model.

In [4]:
game = ch.FlipGame(vectorizer, model)
flip = game.flip            # the one-line version of the game (section 3)
game.flip("I love this sunny day")

👀 **How to read it:** each chip is a word **as the model reads it** (cleaned, lower case). Green = pushes towards POSITIVE, red = towards NEGATIVE; the number is the push. A dashed chip is a word the model never saw: it is ignored. The gauge is the model's verdict, P(positive); the **flip line** is 50 %.

<a id="sec-2"></a>
# 2. 🎚️ Play!

Pick a level, then edit the sentence in the box. Everything updates as you type. Your best flip of each level is saved in the scoreboard at the bottom. **Next level ▶** when you are happy, **↺ Reset** to start the level again.

> If you only see a line like `VBox(children=…)` and no game, your Jupyter cannot show widgets: use section 3, it is the same game in one line of code.

In [5]:
game.play()

<a id="sec-3"></a>
# 3. ⌨️ No widgets? Play with one line of code

Same game, same scoring, without the live updates: `flip("your sentence", level=1)`. In Liar mode, add `honest=True` (your word of honour that a human still reads the sentence as happy). `game.intro(level)` shows the start card of a level.

In [6]:
game.intro(1)

In [7]:
# ✏️ YOUR TURN: (level, your sentence) · each line is one try, the best one per level counts
MY_TRIES = [
    (1, "I do not love this sunny day"),
    (2, "this is the worst headache ever lol"),
]
for level, sentence in MY_TRIES:
    display(flip(sentence, level=level))

👀 **What to notice:** two swaps (*do*, *not*) and the model is **still** 62 % positive: *not* is red, but it is a lonely red word next to a big green *love*. And *lol* is green (+), so it made the headache sentence a bit *more* hopeful, not enough. The model does not know that *not* reverses *love*: it just adds up pushes.

<a id="sec-4"></a>
# 4. 🏆 Your scorecard
Re-run this cell whenever you want: it shows your best flip of each level (from the widget and from `flip`). Copy the yellow line to share your result.

In [8]:
game.scorecard()

<a id="sec-5"></a>
# 5. 🔍 Why did it flip?

> 🧠 **How the model decides**
> 1. The sentence becomes one TF-IDF number per known word (bigger for rare words; all numbers scaled so the sentence has length 1).
> 2. Each word's push = its TF-IDF number × its **weight** (learned from the tweets).
> 3. Score = **bias** + the sum of the pushes → a probability with the sigmoid. Above 50 %: POSITIVE.
> - **Analogy:** a tug of war where every word pulls with a fixed strength, whatever its neighbours say.
> - **Limits:** no word order, no context: *"not good"* is just *not* (red) + *good* (green).

Here are some of the model's weights. Some make sense, some are… absurd:

In [9]:
game.word_board(["thanks", "love", "happy", "great", "sad", "sick", "miss", "hate", "not", "no"])

In [10]:
game.word_board(["worry", "bmw", "free", "wine", "lol", "!", "exam", "work", "cold", "headache", "observer", "lit"])

👀 **The absurd ones, explained:**
- ***worry*** is one of the most **positive** words. On Twitter people write *"don't worry"*, *"no worries"*: the model never sees the *don't*, only that *worry* comes with friendly tweets. So *"the worst worry ever"* can come out positive (that is the Level 2 secret 🤫).
- ***exam***, ***work***, ***cold*** are negative: tweets about exams and work are mostly complaints. The model learned the **topic**, not the opinion: *"my exam went great today"* gets only 25 % positive!
- ***bmw*** positive, ***observer*** negative: a few tweets that happened to be in the data (an ad campaign, a news story). A word seen 20 times can get a big weight: that is **overfitting to rare words**.
- ***sick*** is the most negative word of all, so *"this phone is sick!"* (slang for *amazing*) is NEGATIVE. That is your Liar-mode trick: humans read the **context**, the model reads one word.

And negation, the classic trap:

In [11]:
display(game.flip("not bad at all"), game.flip("I could not be happier"))

*"not bad at all"* is a happy sentence made of two negative words, so the model is about 97 % sure it is NEGATIVE. *"I could not be happier"* survives (just) because *happier* pulls harder than *not*. Neither verdict comes from understanding the negation.

Three more tricks you may have found:
- **Dilution:** the TF-IDF numbers are scaled so the whole sentence has length 1. Adding neutral words makes every other word's push **smaller**, so the score drifts towards the bias. Long sentences are harder to push, which is why the Boss is a long sentence.
- **Unknown words** (dashed chips) push nothing: new slang (*"slay"*, *"yeet"*) is invisible to a model trained in 2009.
- **Punctuation counts:** `!` is a positive word for this model (+2 weight), `?` slightly negative.

| What you did in the game | Why it worked | Back to |
|---|---|---|
| One new word flipped a 99 % sentence | a linear model just **adds** pushes; one big weight can outweigh several small ones | notebook 02 · logistic regression |
| *not*, *no*, *never* did not reverse the meaning | a bag of words has **no order**: *not* is just another (negative) word | notebook 01 · bag of words |
| *sick*, slang, sarcasm fooled it while a human understood | the meaning comes from **context**; transformers read context, a bag of words does not | notebook 02 · transformers |
| *worry*, *exam*, *bmw* behaved strangely | the model learns **correlations** in its data, including topics and accidents | notebook 02 · top coefficients |

Inputs written on purpose to fool a model are called **adversarial examples**. Here you can *see* every push, so finding them is a puzzle; for a transformer it is harder, but people still find them.

## 💬 Discussion
1. Your best Level 1 flip probably still reads as a positive sentence to a human. What does that say about trusting the model's 90 %+ confidence?
2. A company uses this model to flag angry customers. A customer writes *"not bad at all, I could not be happier"*. What happens, and what would you change?
3. How would you make the model resist your best trick? Think about the data, the features and the model.

<details>
<summary>🧑‍🏫 Instructor guide</summary>

1. Confidence is not correctness: the probability only says how far the score is from the flip line, in the model's own (very limited) view. A model can be 99 % sure and wrong; calibration and error analysis on real data matter more than the number on one sentence.
2. The model says NEGATIVE (about 81 %: *not* twice, *bad*), so a happy customer is flagged as angry: a false alarm. Fixes: word pairs (bigrams, *"not bad"* becomes its own feature, as in notebook 02), a negation rule like the lexicon of notebook 02, or a transformer that reads context; and test the model on a checklist of tricky sentences before deploying.
3. **Data:** add examples with negation, slang and sarcasm (or the players' adversarial sentences, labelled by humans) and retrain. **Features:** bigrams, a minimum word count so rare words like *bmw* do not get huge weights. **Model:** stronger regularisation (smaller `C`), or a pretrained transformer. Attackers will find new tricks: robustness is a process, not a setting.

Possible solutions (do not show them before the game!): L1 *"I love this sad day"* (1) · L2 *"this is the worst worry ever"* or *"…worst thanks ever"* (1) · L3 *"this new phone is sick"* or *"…is lit"* (1, a human still reads it as happy; *lit* only just makes it, 50.1 % negative) · L4 add *worry*, *enjoy* and *loving* anywhere (3) · L5 replace three of *happy*, *thank*, *love* with *sad*, *depressing*, *sick* (3).
</details>

<a id="sec-6"></a>
# 6. 📌 Summary

| Idea | In one line |
|---|---|
| Linear text model | Score = bias + one push per word (TF-IDF × weight) |
| Word pushes | You can read exactly why a linear model decided: that is its big advantage |
| Bag-of-words blind spots | No order, no context: negation, slang and sarcasm fool it |
| Learned correlations | Weights reflect the training data, including topics (*exam*) and accidents (*bmw*) |
| Adversarial example | A small change, written on purpose, that flips a model's decision |

**✅ Checklist:**
- [ ] I flipped at least 3 levels.
- [ ] I can explain why *"not bad"* is negative for this model.
- [ ] I found a sentence a human reads as happy and the model calls negative.
- [ ] I can explain why *worry* is a positive word here.

<a id="sec-7"></a>
# 7. 📚 Learn more

> 📚 **Learn more**
> - Ribeiro et al. (2020). [*Beyond Accuracy: Behavioral Testing of NLP Models with CheckList*](https://aclanthology.org/2020.acl-main.442/): test NLP models with negations, typos and swaps, exactly like this game.
> - Jin et al. (2020). [*Is BERT Really Robust? A Strong Baseline for Natural Language Attack on Text Classification*](https://arxiv.org/abs/1907.11932): word-swap attacks, automated.
> - Goodfellow, Shlens & Szegedy (2015). [*Explaining and Harnessing Adversarial Examples*](https://arxiv.org/abs/1412.6572), the paper that made "fooling a model" a research field.
> - scikit-learn: [Working with text data](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html)

---
<div align="center">

⬅️ [Previous: 02.5 · Generative LLMs](../02_project_tweets/02.5_Tweets_Generative_LLMs.ipynb) &nbsp;·&nbsp; 🏠 [Lesson 03](../../README.md) &nbsp;·&nbsp; [Next: 03 · Computer Vision basics: Fashion-MNIST](../../02_computer_vision/01_mini_project_fashion_mnist/03_Computer_Vision_Basics_Fashion_MNIST.ipynb) ➡️

**The AI Learning Hub** · Lesson 03 — Applied AI: unstructured data<br>
MIT License

</div>